# 04 — SAM/Vicon validation

Main local working notebook. Reuses `src.comparison_engine` for metrics and `src.plotting` for visualization. **Already-aligned CSVs with embedded Vicon** are legacy exploratory comparisons, not independent verification of synchronization.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Run Jupyter from the repository or notebooks/ folder")
sys.path.insert(0, str(ROOT))
DATA = ROOT / "lunge_sam_out"
RESULTS = ROOT / "results"
print(ROOT, DATA.exists())

In [ ]:
from src.comparison_engine import compare_angles
from src.plotting import plot_joint_overlays, plot_joint_errors, plot_camera_comparison
files = sorted(DATA.glob("*_joint_angles.csv"))
print(f"Found {len(files)} joint-angle CSV files")
for i, f in enumerate(files):
    print(i, f.name)

In [ ]:
SELECTED = 0
if files:
    csv_path = files[SELECTED]
    df = pd.read_csv(csv_path)
    display(df.head())
    print("Columns:", list(df.columns))
else:
    df = None
    print("Export/download pre-aligned joint angle CSV files into lunge_sam_out/")

In [ ]:
if df is not None:
    import re
    match = re.match(r"(s[0-9]+)_lunge_(cam[0-9]+)_joint_angles[.]csv", csv_path.name, re.I)
    subject, camera = (match.group(1).upper(), match.group(2)) if match else ("unknown", "unknown")
    out_dir = RESULTS / subject / "lunge_01" / camera
    if "time_ms_aligned" not in df or not any(col.startswith("vicon_") for col in df):
        raise ValueError("Expected pre-aligned CSV with embedded Vicon. For raw inputs use independent sync JSON and Vicon CSV.")
    metrics = compare_angles(csv_path, out_dir, subject=subject, trial="lunge_01", camera=camera, allow_legacy_embedded_reference=True, plot=False)
    display(metrics)

In [ ]:
if df is not None:
    display(plot_joint_overlays(df, title=f"{subject} {camera}: SAM vs Vicon"))
    display(plot_joint_errors(df, title=f"{subject} {camera}: SAM − Vicon"))

In [ ]:
if len(files) >= 2:
    first_subject = files[0].name.split("_")[0]
    same_subject = [f for f in files if f.name.startswith(first_subject + "_")]
    if len(same_subject) >= 2:
        camera_frames = {p.name.split("_")[2]: pd.read_csv(p) for p in same_subject}
        display(plot_camera_comparison(camera_frames, joint="knee", method="geo"))

### Scientific checks
The imported CSVs may already be filtered and interpolated. Do not shift again. Document anatomical signs/definitions, and separately report raw versus independently calibrated error. For real independent validation, use raw SAM `time_ms`, reviewed `sync.json`, and standalone Vicon `time_ms`/angle CSV. Aggregate inference across reps/trials/subjects rather than treating all samples as independent.